# Relative Flux Light Curves — Colour-coded by a Third Variable

Variant of `04_relativeFlux.ipynb`.  
The relative flux ratio is displayed with **filled markers whose colour encodes a
third variable** chosen by the user (default: **airmass** = 1/cos(zenith), computed
from the zenith-angle column stored in the visit metadata — angles are in **degrees**
and converted to radians before use).  
A **shared vertical colour-bar** is placed on the right of the row of subplots;
it does not compress the plot area.

Layout per DIA object:
```
  u  |  g  |  r  |  i  |  z  |  y  |  all-bands  ‖ cbar
```
- **x bottom** : Δt [days] from first alert (global t₀ across all bands)
- **x top**    : calendar date (YYYY-MM-DD)
- **y**        : psfFlux(t) / median(psfFlux)  [or apFlux]
- **marker**   : band-specific filled symbol
- **colour**   : third variable (e.g. airmass) — colourmap `jet`
- **colour-bar**: vertical strip, right of figure, shared across all subplots

Three series of figures are produced:
1. `psfFlux` / median  — colour = third variable
2. `apFlux`  / median  — colour = third variable  (skipped if apFlux absent)
3. PSF vs AP overlay   — colour = third variable, PSF = filled marker, AP = open triangle

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab / IN2P3 / CNRS
- **Last update:** 2026-04-01
- **Subject:** Fink alert broker — Rubin LSST photometric calibration check

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas     version : {pd.__version__}")
print(f"numpy      version : {np.__version__}")
print(f"matplotlib version : {mpl.__version__}")

In [ ]:
# Enable interactive matplotlib backend (zoom/pan) when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ───────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_04b"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"  # input: written by notebook 03
DIR_FIGS = f"figs_{NB_TAG}"  # output figures for this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Source files (butler preferred; fallback to consdb) ────────────────────────
FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")

# ── Number of top-ranked DIA objects to plot ───────────────────────────────────
N_TOP = 20  # <── change here

# ── Band definitions ───────────────────────────────────────────────────────────
BANDS = list("ugrizy")
# Each band gets its own filled marker shape so bands are distinguishable
# even in black-and-white prints and in the combined panel.
BAND_MARKERS = {
    "u": "o",  # circle
    "g": "s",  # square
    "r": "^",  # triangle up
    "i": "D",  # diamond
    "z": "v",  # triangle down
    "y": "P",  # plus (filled)
}
# Fallback per-band colours (used only for legend patches in combined panel)
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Third variable (colour axis) ───────────────────────────────────────────────
# Options:
#   'airmass_computed'  : compute X = 1/cos(zenith_rad)  [recommended]
#   any column name present in the parquet, e.g. 'airmass', 'seeing', 'sky_bg'
# If the chosen column is not found, airmass_computed is used as fallback.
COLOR_VAR = "airmass_computed"  # <── change here

# Zenith-angle column candidates (values expected in DEGREES).
# Butler typical names : 'zd', 'zenithAngle'
# consDb  typical names : 'zenith_distance', 'zenith_angle'
# First match wins; angles are converted with np.deg2rad() before cos().
ZENITH_CANDIDATES = ["zd", "zenithAngle", "zenith_distance", "zenith_angle"]

# ── Colourmap ──────────────────────────────────────────────────────────────────
# 'jet' : blue (low airmass) → cyan → green → yellow → red (high airmass).
# Maximum perceptual contrast across the typical airmass range [1.0, 2.5].
# Alternatives: 'turbo' (similar but perceptually uniform), 'rainbow', 'RdYlBu_r'
CMAP_NAME = "jet"  # <── change here

# ── Figure layout constants ────────────────────────────────────────────────────
# The colour-bar is a VERTICAL strip on the RIGHT.
# FIG_RIGHT leaves space for it; CBAR_WIDTH controls its width (figure fraction).
# FIG_RIGHT  = 0.88   # subplots end at 88 % of figure width
FIG_RIGHT = 0.9  # subplots end at 90 % of figure width
CBAR_LEFT = 0.905  # colour-bar starts at 90.5 %
# CBAR_LEFT  = 0.90  # colour-bar starts at 90 %
# CBAR_WIDTH = 0.018  # thin vertical strip
CBAR_WIDTH = 0.005  # thin vertical strip

# ── Matplotlib global settings ─────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Alert column names ─────────────────────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_AP = "r:apFlux"
COL_APERR = "r:apFluxErr"


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")
print(f"COLOR_VAR        : {COLOR_VAR}")
print(f"CMAP_NAME        : {CMAP_NAME}")

## 2. Load data

In [ ]:
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file : {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file : {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found. "
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Source label : {src_label}")

## 3. Schema inspection & column validation

In [ ]:
print("Columns in loaded table:")
print(df.dtypes.to_string())

In [ ]:
required_cols = [COL_OBJ, COL_MJD, COL_BAND, COL_PSF, COL_PSFERR]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

has_ap = (COL_AP in df.columns) and (COL_APERR in df.columns)
if not has_ap:
    print("WARNING: apFlux / apFluxErr columns not found — AP plots will be skipped.")

print("All required columns present.")
print(f"apFlux available : {has_ap}")

## 4. Build the colour variable — airmass from zenith angle

**Airmass** = 1 / cos(z)  where z is the zenith distance in **radians**.  
The parquet stores z in **degrees**, so we apply `np.deg2rad()` first.

In [ ]:
COL_COLOR = "_color_var"  # internal column name

if COLOR_VAR == "airmass_computed" or COLOR_VAR not in df.columns:
    # Auto-detect zenith-angle column
    zenith_col = None
    for candidate in ZENITH_CANDIDATES:
        if candidate in df.columns:
            zenith_col = candidate
            break
    if zenith_col is None:
        for col in df.columns:
            if "zenith" in col.lower():
                zenith_col = col
                break

    if zenith_col is not None:
        print(f"Zenith-angle column : '{zenith_col}'  (unit: degrees)")
        # degrees -> radians -> airmass
        zenith_rad = np.deg2rad(df[zenith_col].astype(float).values)
        cos_z = np.cos(zenith_rad)
        cos_z = np.where(cos_z > 0.01, cos_z, np.nan)  # guard zenith >= 89 deg
        df[COL_COLOR] = 1.0 / cos_z
        color_label = f"Airmass  X = 1/cos(z)   ['{zenith_col}' in deg]"
        print(f"Airmass range : {df[COL_COLOR].min():.3f} - {df[COL_COLOR].max():.3f}")
    else:
        print("WARNING: No zenith-angle column found. Falling back to MJD.")
        df[COL_COLOR] = df[COL_MJD].astype(float)
        color_label = "MJD"
else:
    df[COL_COLOR] = df[COLOR_VAR].astype(float)
    color_label = COLOR_VAR
    print(f"Using column '{COLOR_VAR}'.  Range : {df[COL_COLOR].min():.4g} - {df[COL_COLOR].max():.4g}")

print(f"Colour label : {color_label}")

## 5. Rank DIA objects by number of alerts (decreasing)

In [ ]:
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)
print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP} by alert count:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)
print(f"Rows kept for top-{N_TOP} objects : {len(df_top):,}")

## 6. Global colour normalisation (shared across all figures)

In [ ]:
cvar_all = df_top[COL_COLOR].dropna().values
cvar_min = float(np.nanpercentile(cvar_all, 1))  # clip outliers
cvar_max = float(np.nanpercentile(cvar_all, 99))

CMAP = cm.get_cmap(CMAP_NAME)
CNORM = mcolors.Normalize(vmin=cvar_min, vmax=cvar_max)
SM = cm.ScalarMappable(cmap=CMAP, norm=CNORM)
SM.set_array([])  # required by matplotlib for standalone colourbar

print(f"Global colour range (1-99th pct) : [{cvar_min:.4g},  {cvar_max:.4g}]")
print(f"Colourmap : {CMAP_NAME}")

## 7. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """Convert MJD floats to 'YYYY-MM-DD' strings (astropy, TAI scale)."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_relative_flux(flux, flux_err):
    """
    ratio = flux / median(flux),  ratio_err = flux_err / median(flux).

    Median computed only on finite positive values.

    Returns
    -------
    ratio, ratio_err, flux_median, sigma_ratio
    """
    f = np.asarray(flux, dtype=float)
    fe = np.asarray(flux_err, dtype=float)
    finite_mask = np.isfinite(f) & (f > 0)
    if finite_mask.sum() == 0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), np.nan, np.nan
    f_med = float(np.median(f[finite_mask]))
    if f_med == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), 0.0, np.nan
    ratio = f / f_med
    ratio_err = fe / f_med
    sigma_ratio = float(np.nanstd(ratio[finite_mask]))
    return ratio, ratio_err, f_med, sigma_ratio


def _add_date_axis(ax, dt_array, t0_mjd):
    """
    Secondary x-axis on top of *ax* with 'YYYY-MM-DD' labels.
    At most 5 ticks, spaced to the data range.
    """
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def _scatter_with_errorbars(ax, x, y, yerr, cvals, marker, label=None):
    """
    Draw grey error bars first, then coloured filled markers on top.

    The colour of each marker is determined by its cvals entry via CMAP/CNORM.
    Error bars are neutral grey so the colourmap is readable regardless of cmap.

    Parameters
    ----------
    ax     : Axes
    x      : array  delta-t [days]
    y      : array  ratio values
    yerr   : array  ratio 1-sigma uncertainties
    cvals  : array  third-variable values (mapped via CNORM -> CMAP)
    marker : str    matplotlib marker code (filled)
    label  : str or None  legend entry
    """
    # Grey error bars (no marker cap colour confusion)
    ax.errorbar(
        x,
        y,
        yerr=yerr,
        fmt="none",
        ecolor="0.50",
        elinewidth=0.7,
        capsize=2,
        alpha=0.65,
        zorder=2,
    )
    # Coloured filled markers
    ax.scatter(
        x,
        y,
        c=cvals,
        cmap=CMAP,
        norm=CNORM,
        s=30,
        marker=marker,
        edgecolors="none",
        alpha=0.92,
        zorder=3,
        label=label,
    )


def add_shared_colorbar(fig, label):
    """
    Add a **vertical** colour-bar on the right side of the figure.

    The colour-bar occupies a thin strip at x = [CBAR_LEFT, CBAR_LEFT+CBAR_WIDTH]
    and spans 20-80 % of the figure height to stay compact and not overpower the
    subplots.  The subplots must already have been laid out with right = FIG_RIGHT
    (done by the caller via fig.subplots_adjust).

    Parameters
    ----------
    fig   : Figure
    label : str  colour-bar axis label
    """
    cbar_ax = fig.add_axes([CBAR_LEFT, 0.20, CBAR_WIDTH, 0.60])
    cb = fig.colorbar(SM, cax=cbar_ax, orientation="vertical")
    cb.set_label(label, fontsize=9, labelpad=6)
    cb.ax.tick_params(labelsize=8)


print("Utility functions defined.")

## 8. Main plotting functions

In [ ]:
def plot_relflux_coloured(obj_id, df_obj, flux_col, flux_err_col, flux_label, axes):
    """
    Relative flux per band with colour = third variable.

    Parameters
    ----------
    obj_id       : int/str
    df_obj       : DataFrame  all rows for this object, sorted by MJD
    flux_col     : str        'r:psfFlux' or 'r:apFlux'
    flux_err_col : str
    flux_label   : str        e.g. 'psfFlux'
    axes         : list of 7 Axes  [u, g, r, i, z, y, all]

    Returns
    -------
    t0_date : str  ISO date of the first alert
    """
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    ax_all = axes[-1]

    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)

        if len(df_b) == 0:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            continue

        dt = df_b[COL_MJD].values - t0_mjd
        cvals = df_b[COL_COLOR].values
        ratio, ratio_err, _, sigma = compute_relative_flux(df_b[flux_col].values, df_b[flux_err_col].values)
        marker = BAND_MARKERS.get(band, "o")

        # Individual band panel
        _scatter_with_errorbars(ax, dt, ratio, ratio_err, cvals, marker)
        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band} {marker} sigma={sigma:.3f}", fontsize=9)
        ax.set_ylabel(f"{flux_label} / median", fontsize=8)
        _add_date_axis(ax, dt, t0_mjd)

        # Combined panel
        _scatter_with_errorbars(
            ax_all, dt, ratio, ratio_err, cvals, marker, label=f"{band} (sigma={sigma:.3f})"
        )

    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands", fontsize=9)
    ax_all.set_ylabel(f"{flux_label} / median", fontsize=8)
    ax_all.legend(fontsize=7, ncol=3, loc="best")
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    return t0_date


def plot_psf_vs_ap_coloured(obj_id, df_obj, axes):
    """
    Overlay PSF (filled marker) and AP (open triangle) relative fluxes,
    both colour-coded by the third variable.

    Returns
    -------
    t0_date : str
    """
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    ax_all = axes[-1]

    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)

        if len(df_b) == 0:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            continue

        dt = df_b[COL_MJD].values - t0_mjd
        cvals = df_b[COL_COLOR].values
        marker = BAND_MARKERS.get(band, "o")

        # PSF ratio — filled marker
        psf_ratio, psf_err, _, sigma_psf = compute_relative_flux(
            df_b[COL_PSF].values, df_b[COL_PSFERR].values
        )
        _scatter_with_errorbars(ax, dt, psf_ratio, psf_err, cvals, marker, label=f"psf sigma={sigma_psf:.3f}")

        # AP ratio — open triangle, same colour mapping on the edge
        if has_ap:
            ap_ratio, ap_err, _, sigma_ap = compute_relative_flux(df_b[COL_AP].values, df_b[COL_APERR].values)
            ax.errorbar(
                dt,
                ap_ratio,
                yerr=ap_err,
                fmt="none",
                ecolor="0.60",
                elinewidth=0.7,
                capsize=2,
                alpha=0.6,
                zorder=2,
            )
            edge_colors = CMAP(CNORM(cvals))
            # ax.scatter(dt, ap_ratio,
            #           c="none", edgecolors=edge_colors,
            #           s=32, marker="^", linewidths=1.0,
            #           alpha=0.85, zorder=3,
            #           label=f"ap   sigma={sigma_ap:.3f}")
            ax.scatter(
                dt,
                ap_ratio,
                c="none",
                edgecolors=edge_colors,
                s=32,
                marker=marker,
                linewidths=1.0,
                alpha=0.85,
                zorder=3,
                label=f"ap   sigma={sigma_ap:.3f}",
            )

        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band}", fontsize=9)
        ax.set_ylabel("flux / median", fontsize=8)
        ax.legend(fontsize=7, loc="best")
        _add_date_axis(ax, dt, t0_mjd)

        # Combined panel — PSF filled + AP open
        _scatter_with_errorbars(ax_all, dt, psf_ratio, psf_err, cvals, marker, label=f"{band} psf")
        if has_ap:
            edge_colors = CMAP(CNORM(cvals))
            ax_all.scatter(
                dt,
                ap_ratio,
                c="none",
                edgecolors=edge_colors,
                s=22,
                marker="^",
                linewidths=0.8,
                alpha=0.60,
                zorder=3,
                label=f"{band} ap",
            )

    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands — psf filled / ap open", fontsize=9)
    ax_all.set_ylabel("flux / median", fontsize=8)
    ax_all.legend(fontsize=6, ncol=4, loc="best")
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    return t0_date


print("Plotting functions defined.")

## 9. Loop — PSF relative flux, colour = third variable

The colour-bar is a **vertical strip** placed to the right of the subplots area.
It is built *after* the subplots, using `fig.add_axes` with pre-reserved space
(`right = FIG_RIGHT`), so it never compresses the data panels.

In [ ]:
n_subplots = len(BANDS) + 1  # 6 bands + combined

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_obj)

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.8),
        sharey=False,
        constrained_layout=False,
    )
    # Reserve space on the right for the vertical colourbar.
    # The subplots themselves end at FIG_RIGHT; top is lowered to leave
    # room for the secondary date axis labels.
    fig.subplots_adjust(
        left=0.05,
        right=FIG_RIGHT,
        bottom=0.13,
        top=0.78,
        wspace=0.45,
    )

    t0_date = plot_relflux_coloured(
        obj_id,
        df_obj,
        flux_col=COL_PSF,
        flux_err_col=COL_PSFERR,
        flux_label="psfFlux",
        axes=axes,
    )

    for ax in axes:
        ax.set_xlabel("delta-t [days] from first alert", fontsize=8)

    add_shared_colorbar(fig, label=color_label)

    field = df_obj["field"].unique()[0]

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}({field})  |  N={n_alerts} alerts  |  t0={t0_date}\n"
        f"psfFlux/median  —  colour: {color_label}",
        fontsize=10,
        y=0.98,
    )

    savefig(f"relflux_psf_coloured_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()
    # plt.close(fig)

## 10. Loop — AP relative flux, colour = third variable

In [ ]:
if not has_ap:
    print("apFlux columns not available — skipping AP relative flux plots.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        n_alerts = len(df_obj)

        fig, axes = plt.subplots(
            1,
            n_subplots,
            figsize=(3.2 * n_subplots, 4.8),
            sharey=False,
            constrained_layout=False,
        )
        fig.subplots_adjust(
            left=0.05,
            right=FIG_RIGHT,
            bottom=0.13,
            top=0.78,
            wspace=0.45,
        )

        t0_date = plot_relflux_coloured(
            obj_id,
            df_obj,
            flux_col=COL_AP,
            flux_err_col=COL_APERR,
            flux_label="apFlux",
            axes=axes,
        )

        for ax in axes:
            ax.set_xlabel("delta-t [days] from first alert", fontsize=8)

        add_shared_colorbar(fig, label=color_label)

        field = df_obj["field"].unique()[0]

        fig.suptitle(
            f"rank #{rank + 1}  |  diaObjectId={obj_id}({field})  |  N={n_alerts} alerts  |  t0={t0_date}\n"
            f"apFlux/median  —  colour: {color_label}",
            fontsize=10,
            y=0.98,
        )

        savefig(f"relflux_ap_coloured_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
        plt.show()
        # plt.close(fig)

## 11. Loop — PSF vs AP overlay, colour = third variable

- **Filled band-specific marker** : PSF ratio
- **Open triangle (same jet colour on edge)** : AP ratio

In [ ]:
for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_obj)

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.8),
        sharey=False,
        constrained_layout=False,
    )
    fig.subplots_adjust(
        left=0.05,
        right=FIG_RIGHT,
        bottom=0.13,
        top=0.75,
        wspace=0.45,
    )

    t0_date = plot_psf_vs_ap_coloured(obj_id, df_obj, axes)

    for ax in axes:
        ax.set_xlabel("delta-t [days] from first alert", fontsize=8)

    add_shared_colorbar(fig, label=color_label)

    field = df_obj["field"].unique()[0]

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id} ({field}) |  N={n_alerts} alerts  |  t0={t0_date}\n"
        f"psfFlux/median (filled) vs apFlux/median (open tri)  —  colour: {color_label}",
        fontsize=10,
        y=0.97,
    )

    savefig(f"psf_vs_ap_coloured_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()
    # plt.close(fig)

## 12. Summary table: sigma(ratio) per object & band

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    n_total = len(df_obj)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_alerts": n_total,
        "t0_date": t0_date,
        "color_var_median": round(float(df_obj[COL_COLOR].median()), 4),
    }

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"sigma_psf_{band}"] = np.nan
            row[f"sigma_ap_{band}"] = np.nan
            continue
        _, _, _, sigma_psf = compute_relative_flux(df_b[COL_PSF].values, df_b[COL_PSFERR].values)
        row[f"sigma_psf_{band}"] = round(sigma_psf, 4)
        if has_ap:
            _, _, _, sigma_ap = compute_relative_flux(df_b[COL_AP].values, df_b[COL_APERR].values)
            row[f"sigma_ap_{band}"] = round(sigma_ap, 4)
        else:
            row[f"sigma_ap_{band}"] = np.nan

    records.append(row)

df_summary = pd.DataFrame(records)
print("Summary table — relative-flux scatter + median colour variable:")
display(df_summary)

In [ ]:
summary_path = os.path.join(DIR_FIGS, f"sigma_ratio_coloured_summary_{src_label}.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")